In [23]:
import ee
import folium
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split

# หากใช้ใน Google Colab ต้องติดตั้งไลบรารีก่อน
# !pip install earthengine-api folium matplotlib pandas

# ใน Google Colab ต้องใช้ ee.Authenticate() แทน earthengine authenticate เพราะ Colab ไม่มี CLI เหมือนเครื่อง Local
ee.Authenticate()

# ตรวจสอบและเริ่มต้นการเชื่อมต่อ GEE
try:
    ee.Initialize(project="ee-sakda-451407")
except Exception as e:
    ee.Authenticate() # หากยังไม่ได้ยืนยันตัวตน ให้ทำการยืนยัน
    ee.Initialize(project="ee-sakda-451407") 

In [ ]:
# Step 1: Define Thailand geometry using FAO/GAUL dataset
# thailand = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level0')\
#     .filter(ee.Filter.eq('ADM0_NAME', 'Thailand'))\
#     .geometry()

thailand = ee.Geometry.Polygon([
    [
        [98.5, 19.5], 
        [99.5, 19.5], 
        [99.5, 18.5], 
        [98.5, 18.5], 
        [98.5, 19.5]  
    ]
]);

# Step 2: Load FIRMS VIIRS fire data
firms = ee.ImageCollection('FIRMS')\
    .select('T21')\
    .filterDate('2024-01-01', '2024-12-31')\
    .filterBounds(thailand)

# Step 3: Convert ImageCollection to FeatureCollection of fire points
def create_fire_points(image):
    fire_mask = image.select('T21').gt(0)\
        .set('system:time_start', image.get('system:time_start'))
    vectors = fire_mask.reduceToVectors(
        geometry=thailand,
        scale=375,
        geometryType='centroid',
        labelProperty='fire',
        maxPixels=1e9
    ).filterBounds(thailand)
    return vectors.map(lambda feature: feature.set('system:time_start', image.get('system:time_start')))

fire_points = firms.map(create_fire_points)
flattened_points = fire_points.flatten()

# Step 4: Function to count hotspots by week
def count_hotspots_by_week(fire_points, start_date, end_date):
    start = ee.Date(start_date)
    end = ee.Date(end_date)
    weeks = ee.List.sequence(0, end.difference(start, 'week').floor())
    
    def count_week(week_offset):
        week_start = start.advance(week_offset, 'week')
        week_end = week_start.advance(1, 'week')
        weekly_points = fire_points.filterDate(week_start, week_end)
        count = weekly_points.size()
        return ee.Feature(None, {
            'week': week_start.format('YYYY-MM-dd'),
            'hotspot': count
        })
    
    return ee.FeatureCollection(weeks.map(count_week))

# Step 5: Generate weekly counts
weekly_hotspot_counts = count_hotspots_by_week(flattened_points, '2024-01-01', '2024-12-31')

# Step 6: Export weekly counts to a Pandas DataFrame
weekly_counts_list = weekly_hotspot_counts.getInfo()['features']
df = pd.DataFrame([{
    'week': f['properties']['week'],
    'hotspot': f['properties']['hotspot']
} for f in weekly_counts_list])



In [21]:

# Initialize Folium map
m = folium.Map(location=[15.8700, 100.9925], zoom_start=6)
add_ee_layer(thailand, {'color': 'yellow'}, 'Thailand Boundary')
add_ee_layer(flattened_points, {'color': 'red'}, 'FIRMS Hotspots')
m.add_child(folium.LayerControl())
# m.save('thailand_hotspots.html')
m

In [ ]:
# Step 8: Create a Matplotlib chart
plt.figure(figsize=(12, 6))
plt.bar(df['week_start'], df['hotspot_count'], width=6, color='#ff4500', align='center')
plt.title('Weekly FIRMS Hotspots in Thailand (2024)')
plt.xlabel('Week Start Date')
plt.ylabel('Number of Hotspots')
plt.gca().xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('weekly_hotspot_counts.png')
plt.show()

# Step 9: Print total hotspot count
total_count = flattened_points.size().getInfo()
print(f'Total Number of FIRMS Hotspots in Thailand: {total_count}')

In [ ]:

# Step 7: Prepare data for LSTM
# Ensure DataFrame is sorted by date and handle missing values
df['date'] = pd.to_datetime(df['date'])
df = df.set_index('date').sort_index()
df['hotspot_count'] = pd.to_numeric(df['hotspot_count'], errors='coerce').fillna(0)

# Extract hotspot counts and dates
hotspots = df['hotspot_count'].values
dates = df.index.values

# Normalize the data
scaler = MinMaxScaler()
hotspots_scaled = scaler.fit_transform(hotspots.reshape(-1, 1))

# Create sequences for LSTM
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

sequence_length = 12
X, y = create_sequences(hotspots_scaled, sequence_length)

# Step 8: Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Step 9: Build and train LSTM model
model = Sequential()
model.add(LSTM(64, activation='tanh', return_sequences=True, dropout=0.2, recurrent_dropout=0.2, input_shape=(sequence_length, 1)))
model.add(LSTM(32, activation='tanh', dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')
model.summary()

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

# Step 10: Make predictions
y_pred = model.predict(X_test)

# Inverse transform predictions and actual values
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_inv = scaler.inverse_transform(y_pred)